# Colab runner — unimodal-transformer

Thin launcher: this notebook clones the repo onto the Colab VM and drives the modular `src/` code on a GPU.

**Before running:** `Runtime → Change runtime type → T4 GPU`.

Every cell below runs on the Google VM, not your Mac. The VM is ephemeral — it is wiped when the runtime resets — so you re-run the clone/install cell at the start of each session, and anything you want to keep must be pushed to GitHub or saved to Drive (last section).

## 1. Check the GPU

In [ ]:
!nvidia-smi
import torch
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU — set Runtime > Change runtime type > T4 GPU")

## 2. Clone the repo onto the VM and install deps

Idempotent: clones on first run, `git pull` on re-runs within the same session. Colab preinstalls torch/numpy/scipy/matplotlib, so the install is fast.

In [ ]:
import os

REPO_URL = "https://github.com/slocke3/unimodal-transformer.git"
REPO_DIR = "/content/unimodal-transformer"

if not os.path.exists(REPO_DIR):
    !git clone $REPO_URL $REPO_DIR

%cd $REPO_DIR
!git pull --ff-only
!pip install -q -r requirements.txt
print("\nrepo contents:")
!ls

## 3. Smoke test

Tiny end-to-end run (50 trajectories, small model, 1 epoch) to confirm data → model → train loop executes on the GPU before launching a real run.

In [ ]:
from src.dataset import make_splits
from src.model import DiscreteTrajectoryTransformer
from src.trainer import Trainer, TrainerConfig

train_loader, val_loader, _ = make_splits(
    n_trajectories=50, context_len=20, n_bins=32, traj_len=80,
    batch_size=128, num_workers=2, seed=0,
)
model = DiscreteTrajectoryTransformer(
    n_bins=32, context_len=20, d_model=64, n_heads=2, n_layers=2,
)
cfg = TrainerConfig(max_epochs=1, patience=5, save_dir="outputs/checkpoints")
trainer = Trainer(model, train_loader, val_loader, config=cfg, run_name="smoke")
print("device:", trainer.device)
history = trainer.train()
print("smoke OK | best val loss:", history["best_val_loss"])

## 4. Full training run

Drives the CLI with the base config. Saves checkpoint, history, and resolved config under `outputs/checkpoints/`.

In [ ]:
!python train.py --config configs/base.yaml --run_name base

## 5. Evaluate

Per-r cross-entropy vs Lyapunov, cross-family generalization, and rollout plots → `outputs/figures/`.

In [ ]:
!python evaluate.py \
    --checkpoint outputs/checkpoints/base_best.pt \
    --config outputs/checkpoints/base_config.yaml

## 5b. View figures inline (no Drive trip needed)

`evaluate.py` runs as a subprocess, so it saves PNGs to `outputs/figures/` but can't display them. This cell renders every figure right here in the notebook — so you check results in the tab you're already in, without opening Drive.

In [ ]:
import glob
from IPython.display import Image, display

figs = sorted(glob.glob("outputs/figures/*.png"))
if not figs:
    print("No figures found — run the Evaluate cell first.")
for path in figs:
    print(path)
    display(Image(filename=path))

## 6. Persist outputs to Drive (the VM is ephemeral!)

`outputs/` is gitignored and lives only on the throwaway VM, so it vanishes when the runtime resets. The cell below mounts your Google Drive and copies `checkpoints/` + `figures/` into a **timestamped** folder (so repeated runs don't overwrite each other).

**Getting them onto your Mac:** install **Google Drive for desktop** once, and this folder syncs into a local folder you can open straight from VS Code / Finder — no manual download. Anything in Drive is also reachable from [drive.google.com](https://drive.google.com). Set `RUN_NAME` to match the `--run_name` you trained with.

In [ ]:
import os, shutil, datetime
from google.colab import drive

RUN_NAME   = "base"  # match the --run_name used in training
DRIVE_BASE = "/content/drive/MyDrive/unimodal-transformer-outputs"

drive.mount("/content/drive")

stamp = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
dest  = os.path.join(DRIVE_BASE, f"{RUN_NAME}_{stamp}")
os.makedirs(dest, exist_ok=True)

for sub in ("checkpoints", "figures"):
    src = os.path.join("outputs", sub)
    if os.path.isdir(src) and os.listdir(src):
        shutil.copytree(src, os.path.join(dest, sub), dirs_exist_ok=True)
        print(f"copied {src}/  ->  {dest}/{sub}")
    else:
        print(f"(skip) {src}/ is empty or missing")

print("\nSaved to Drive:", dest)

Quick alternative for grabbing a single file straight to your browser's downloads (no Drive needed):

In [ ]:
# from google.colab import files
# files.download("outputs/figures/base_ce_vs_lyapunov.png")